In [ ]:
!pip install lifelines

In [ ]:
# ============================================================
# Figure 5
# TCGA-LIHC validation of RLP3/RLP4-derived R-loop program
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu, spearmanr
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# ============================================================
# 0. Path config
# ============================================================

PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"

FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
FIG5_DIR = os.path.join(PROJECT_DIR, "05_Figure5_TCGA_LIHC")
os.makedirs(FIG5_DIR, exist_ok=True)

# Figure2输出的NMF基因
NMF_TOP_GENE_FILE = os.path.join(
    FIG2_DIR,
    "Fig2_NMF_top_genes_per_Rloop_program.csv"
)

# 你需要提前准备TCGA-LIHC表达矩阵和临床信息
# 表达矩阵格式建议：第一列为gene，后面每列为TCGA样本
TCGA_EXPR_FILE = os.path.join(
    PROJECT_DIR,
    "00_datacollection",
    "TCGA_LIHC_expression.csv"
)

TCGA_CLIN_FILE = os.path.join(
    PROJECT_DIR,
    "00_datacollection",
    "TCGA_LIHC_clinical.csv"
)

# ============================================================
# 1. Plot setting
# ============================================================

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False

def save_fig(fig, name, w=6, h=5):
    fig.set_size_inches(w, h)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(FIG5_DIR, f"{name}.{ext}"),
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )
    plt.close(fig)
    print("Saved:", name)

In [ ]:
# ============================================================
# 2. Load RLP3 / RLP4 genes from Figure2
# ============================================================

nmf_genes = pd.read_csv(NMF_TOP_GENE_FILE)

rlp3_genes = (
    nmf_genes.loc[nmf_genes["program"] == "RLP3", "gene"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

rlp4_genes = (
    nmf_genes.loc[nmf_genes["program"] == "RLP4", "gene"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("RLP3 genes:", len(rlp3_genes))
print("RLP4 genes:", len(rlp4_genes))
print("RLP3 example:", rlp3_genes[:10])
print("RLP4 example:", rlp4_genes[:10])

In [ ]:
# ============================================================
# 3. Load TCGA-LIHC expression matrix
# ============================================================

expr = pd.read_csv(TCGA_EXPR_FILE)

# 自动识别gene列
gene_col_candidates = ["gene", "Gene", "gene_name", "symbol", "GeneSymbol"]
gene_col = None

for c in gene_col_candidates:
    if c in expr.columns:
        gene_col = c
        break

if gene_col is None:
    gene_col = expr.columns[0]

expr = expr.rename(columns={gene_col: "gene"})
expr["gene"] = expr["gene"].astype(str)

# 去除重复基因，取平均
expr_mat = (
    expr
    .groupby("gene")
    .mean(numeric_only=True)
)

# gene x sample -> sample x gene
expr_t = expr_mat.T.copy()

# TCGA barcode统一为前12位患者ID
expr_t.index = expr_t.index.astype(str).str.slice(0, 12)

# 同一个患者多个样本时取平均
expr_t = expr_t.groupby(expr_t.index).mean()

print("TCGA expression matrix:", expr_t.shape)
expr_t.head()

In [ ]:
# ============================================================
# 4. Calculate RLP3 / RLP4 / combined score in TCGA-LIHC
# ============================================================

def score_gene_set(expr_sample_gene, genes, score_name):
    genes_use = [g for g in genes if g in expr_sample_gene.columns]
    print(score_name, "matched genes:", len(genes_use))

    if len(genes_use) == 0:
        raise ValueError(f"No genes matched for {score_name}")

    X = expr_sample_gene[genes_use].copy()
    X_z = pd.DataFrame(
        StandardScaler().fit_transform(X),
        index=X.index,
        columns=X.columns
    )

    return X_z.mean(axis=1), genes_use

tcga_score = pd.DataFrame(index=expr_t.index)

tcga_score["RLP3_score"], rlp3_matched = score_gene_set(
    expr_t,
    rlp3_genes,
    "RLP3"
)

tcga_score["RLP4_score"], rlp4_matched = score_gene_set(
    expr_t,
    rlp4_genes,
    "RLP4"
)

score_z = pd.DataFrame(
    StandardScaler().fit_transform(tcga_score[["RLP3_score", "RLP4_score"]]),
    index=tcga_score.index,
    columns=["RLP3_score_z", "RLP4_score_z"]
)

tcga_score["RLP3_RLP4_combined_score"] = score_z.mean(axis=1)

median_score = tcga_score["RLP3_RLP4_combined_score"].median()

tcga_score["Rloop_program_group"] = np.where(
    tcga_score["RLP3_RLP4_combined_score"] >= median_score,
    "R-loop-program-high",
    "R-loop-program-low"
)

tcga_score.to_csv(
    os.path.join(FIG5_DIR, "Figure5_TCGA_RLP3_RLP4_scores.csv")
)

tcga_score["Rloop_program_group"].value_counts()

In [ ]:
# ============================================================
# Figure5A
# RLP3/RLP4 score distribution in TCGA-LIHC
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=8, height=4, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Prepare plotting dataframe
# ============================================================

plot_df = tcga_score.sort_values(
    "RLP3_RLP4_combined_score"
).copy()

plot_df["sample_order"] = np.arange(plot_df.shape[0])

# ============================================================
# Create figure
# ============================================================

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    plot_df["sample_order"],
    plot_df["RLP3_RLP4_combined_score"],
    color="black",
    linewidth=1.2
)

ax.scatter(
    plot_df["sample_order"],
    plot_df["RLP3_RLP4_combined_score"],
    c=plot_df["Rloop_program_group"].map({
        "R-loop-program-high": "#1f77b4",
        "R-loop-program-low": "#ff7f0e"
    }),
    s=18,
    alpha=0.85
)

ax.axhline(
    median_score,
    linestyle="--",
    color="grey",
    linewidth=1
)

ax.set_xlabel(
    "TCGA-LIHC samples ordered by combined score",
    fontsize=11
)

ax.set_ylabel(
    "RLP3/RLP4 combined score",
    fontsize=11
)

ax.set_title(
    "RLP3/RLP4-derived R-loop program score in TCGA-LIHC",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10
)

plt.tight_layout()

save_fig(
    fig,
    "Figure5A_TCGA_RLP3_RLP4_score_distribution",
    8,
    4
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# 5. Load clinical data
# ============================================================

clin = pd.read_csv(TCGA_CLIN_FILE)

# 自动识别样本ID列
id_candidates = ["sample", "Sample", "patient", "Patient", "submitter_id", "case_submitter_id", "bcr_patient_barcode"]
id_col = None

for c in id_candidates:
    if c in clin.columns:
        id_col = c
        break

if id_col is None:
    id_col = clin.columns[0]

clin = clin.rename(columns={id_col: "patient_id"})
clin["patient_id"] = clin["patient_id"].astype(str).str.slice(0, 12)

clin = clin.drop_duplicates("patient_id").set_index("patient_id")

data = tcga_score.join(clin, how="inner")

print("Merged TCGA score + clinical:", data.shape)
data.head()

In [ ]:
# ============================================================
# Figure5B
# RLP3/RLP4 combined score: Early vs Advanced stage
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=4.2, height=4.5, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Prepare clinical stage data
# ============================================================

stage_col = "ajcc_pathologic_stage"

tmp = data[[stage_col, "RLP3_RLP4_combined_score"]].dropna().copy()
tmp[stage_col] = tmp[stage_col].astype(str)

tmp = tmp[
    ~tmp[stage_col].isin([
        "nan",
        "NaN",
        "not reported",
        "Not Reported",
        "None"
    ])
].copy()

# ============================================================
# Simplify stage
# ============================================================

def simplify_stage(x):

    x = str(x).upper()

    if "STAGE I" in x and "STAGE II" not in x and "STAGE III" not in x and "STAGE IV" not in x:
        return "Early stage"

    elif "STAGE II" in x and "STAGE III" not in x and "STAGE IV" not in x:
        return "Early stage"

    elif "STAGE III" in x:
        return "Advanced stage"

    elif "STAGE IV" in x:
        return "Advanced stage"

    else:
        return np.nan

tmp["Stage_group"] = tmp[stage_col].apply(simplify_stage)
tmp = tmp.dropna(subset=["Stage_group"]).copy()

tmp["Stage_group"] = pd.Categorical(
    tmp["Stage_group"],
    categories=["Early stage", "Advanced stage"],
    ordered=True
)

print(tmp["Stage_group"].value_counts())

# ============================================================
# Mann-Whitney U test
# ============================================================

early = tmp.loc[
    tmp["Stage_group"] == "Early stage",
    "RLP3_RLP4_combined_score"
]

advanced = tmp.loc[
    tmp["Stage_group"] == "Advanced stage",
    "RLP3_RLP4_combined_score"
]

stat, pval = mannwhitneyu(
    early,
    advanced,
    alternative="two-sided"
)

print("Mann-Whitney U test:")
print("U =", stat)
print("P =", pval)

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(4.2, 4.5))

sns.boxplot(
    data=tmp,
    x="Stage_group",
    y="RLP3_RLP4_combined_score",
    order=["Early stage", "Advanced stage"],
    color="white",
    linewidth=1.2,
    fliersize=0,
    ax=ax
)

sns.stripplot(
    data=tmp,
    x="Stage_group",
    y="RLP3_RLP4_combined_score",
    order=["Early stage", "Advanced stage"],
    color="black",
    size=3.5,
    alpha=0.55,
    jitter=0.22,
    ax=ax
)

# ============================================================
# Add P value
# ============================================================

ymax = tmp["RLP3_RLP4_combined_score"].max()
ymin = tmp["RLP3_RLP4_combined_score"].min()
yrange = ymax - ymin

y = ymax + yrange * 0.12

ax.plot(
    [0, 0, 1, 1],
    [y, y + yrange * 0.04, y + yrange * 0.04, y],
    color="black",
    linewidth=1
)

ax.text(
    0.5,
    y + yrange * 0.06,
    f"P = {pval:.3g}",
    ha="center",
    va="bottom",
    fontsize=11
)

# ============================================================
# Labels and title
# ============================================================

ax.set_xlabel("")

ax.set_ylabel(
    "RLP3/RLP4 combined score",
    fontsize=11
)

ax.set_title(
    "RLP3/RLP4 score in early versus advanced LIHC",
    fontsize=13,
    fontweight="bold"
)

# ============================================================
# Publication-style formatting
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10
)

# 给P值上方留空间
ax.set_ylim(
    ymin - yrange * 0.08,
    y + yrange * 0.18
)

plt.tight_layout()

# ============================================================
# Save figure
# ============================================================

save_fig(
    fig,
    "Figure5B_TCGA_RLP3_RLP4_score_early_vs_advanced_stage",
    4.2,
    4.5
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# Save Figure5B statistics
# ============================================================

fig5b_stat = pd.DataFrame({
    "comparison": ["Early stage vs Advanced stage"],
    "early_n": [early.shape[0]],
    "advanced_n": [advanced.shape[0]],
    "early_median": [early.median()],
    "advanced_median": [advanced.median()],
    "mannwhitney_U": [stat],
    "p_value": [pval]
})

fig5b_stat.to_csv(
    os.path.join(FIG5_DIR, "Figure5B_early_vs_advanced_stage_statistics.csv"),
    index=False
)

fig5b_stat

In [ ]:
# ============================================================
# Optional Figure5B2
# RLP3/RLP4 combined score across tumor grade
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=4.8, height=4.5, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Prepare tumor grade data
# ============================================================

grade_col = "tumor_grade"

tmp_grade = data[[grade_col, "RLP3_RLP4_combined_score"]].dropna().copy()
tmp_grade[grade_col] = tmp_grade[grade_col].astype(str)

tmp_grade = tmp_grade[
    ~tmp_grade[grade_col].isin([
        "nan",
        "NaN",
        "not reported",
        "Not Reported",
        "None"
    ])
].copy()

grade_order = ["G1", "G2", "G3", "G4"]
grade_order = [
    g for g in grade_order 
    if g in tmp_grade[grade_col].unique()
]

print("Tumor grade sample counts:")
print(tmp_grade[grade_col].value_counts())

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(4.8, 4.5))

sns.boxplot(
    data=tmp_grade,
    x=grade_col,
    y="RLP3_RLP4_combined_score",
    order=grade_order,
    color="white",
    linewidth=1.2,
    fliersize=0,
    ax=ax
)

sns.stripplot(
    data=tmp_grade,
    x=grade_col,
    y="RLP3_RLP4_combined_score",
    order=grade_order,
    color="black",
    size=3.5,
    alpha=0.55,
    jitter=0.22,
    ax=ax
)

# ============================================================
# Labels and title
# ============================================================

ax.set_xlabel(
    "Tumor grade",
    fontsize=11
)

ax.set_ylabel(
    "RLP3/RLP4 combined score",
    fontsize=11
)

ax.set_title(
    "RLP3/RLP4 score across tumor grade",
    fontsize=13,
    fontweight="bold"
)

# ============================================================
# Publication-style formatting
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10
)

plt.tight_layout()

# ============================================================
# Save figure
# ============================================================

save_fig(
    fig,
    "Figure5B2_TCGA_RLP3_RLP4_score_tumor_grade",
    4.8,
    4.5
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# Figure5B beautiful version
# Adobe Illustrator editable text
# ============================================================

import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=4.3, height=4.8, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(4.3, 4.8))

palette = {
    "Early stage": "#5DA5DA",
    "Advanced stage": "#F17CB0"
}

sns.violinplot(
    data=tmp,
    x="Stage_group",
    y="RLP3_RLP4_combined_score",
    order=["Early stage", "Advanced stage"],
    palette=palette,
    inner=None,
    cut=0,
    linewidth=1,
    alpha=0.8,
    ax=ax
)

sns.boxplot(
    data=tmp,
    x="Stage_group",
    y="RLP3_RLP4_combined_score",
    order=["Early stage", "Advanced stage"],
    width=0.22,
    showcaps=True,
    boxprops={
        "facecolor": "white",
        "zorder": 3
    },
    showfliers=False,
    whiskerprops={
        "linewidth": 1
    },
    medianprops={
        "color": "black",
        "linewidth": 1.5
    },
    ax=ax
)

sns.stripplot(
    data=tmp,
    x="Stage_group",
    y="RLP3_RLP4_combined_score",
    order=["Early stage", "Advanced stage"],
    color="black",
    size=2.8,
    alpha=0.45,
    jitter=0.22,
    ax=ax
)

# ============================================================
# P value
# ============================================================

ymax = tmp["RLP3_RLP4_combined_score"].max()
ymin = tmp["RLP3_RLP4_combined_score"].min()
yrange = ymax - ymin

y = ymax + yrange * 0.08

ax.plot(
    [0, 0, 1, 1],
    [y, y + yrange * 0.03, y + yrange * 0.03, y],
    color="black",
    linewidth=1
)

ax.text(
    0.5,
    y + yrange * 0.05,
    f"P = {pval:.3g}",
    ha="center",
    va="bottom",
    fontsize=11
)

# ============================================================
# Labels and title
# ============================================================

ax.set_xlabel("")

ax.set_ylabel(
    "RLP3/RLP4 combined score",
    fontsize=11
)

ax.set_title(
    "RLP3/RLP4-derived R-loop program in LIHC progression",
    fontsize=13,
    fontweight="bold"
)

# ============================================================
# Formatting
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(False)

ax.tick_params(
    axis="both",
    labelsize=10
)

ax.set_ylim(
    ymin - yrange * 0.08,
    y + yrange * 0.16
)

plt.tight_layout()

# ============================================================
# Save figure
# ============================================================

save_fig(
    fig,
    "Figure5B_TCGA_RLP3_RLP4_score_early_vs_advanced_stage_beautiful",
    4.3,
    4.8
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# Figure5C
# Survival analysis
# Adobe Illustrator editable text
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=5.2, height=4.4, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Automatically identify survival columns
# ============================================================

time_candidates = [
    "OS.time",
    "OS_time",
    "days_to_death",
    "days_to_last_follow_up",
    "time"
]

status_candidates = [
    "OS",
    "OS_status",
    "vital_status",
    "status"
]

time_col = None
status_col = None

for c in time_candidates:
    if c in data.columns:
        time_col = c
        break

for c in status_candidates:
    if c in data.columns:
        status_col = c
        break

print("Survival time column:", time_col)
print("Survival status column:", status_col)

# ============================================================
# Survival analysis
# ============================================================

surv = data.copy()

if time_col is not None and status_col is not None:

    surv["surv_time"] = pd.to_numeric(
        surv[time_col],
        errors="coerce"
    )

    if surv[status_col].dtype == "object":

        surv["surv_event"] = (
            surv[status_col]
            .astype(str)
            .str.lower()
            .str.strip()
            .map({
                "dead": 1,
                "deceased": 1,
                "1": 1,
                "true": 1,
                "yes": 1,
                "event": 1,
                "alive": 0,
                "living": 0,
                "0": 0,
                "false": 0,
                "no": 0,
                "censored": 0
            })
        )

    else:

        surv["surv_event"] = pd.to_numeric(
            surv[status_col],
            errors="coerce"
        )

    surv = surv[
        [
            "surv_time",
            "surv_event",
            "Rloop_program_group",
            "RLP3_RLP4_combined_score"
        ]
    ].dropna().copy()

    surv = surv[surv["surv_time"] > 0].copy()

    print("Survival sample number:")
    print(surv["Rloop_program_group"].value_counts())

    high = surv[
        surv["Rloop_program_group"] == "R-loop-program-high"
    ].copy()

    low = surv[
        surv["Rloop_program_group"] == "R-loop-program-low"
    ].copy()

    # ========================================================
    # Kaplan-Meier plot
    # ========================================================

    kmf = KaplanMeierFitter()

    fig, ax = plt.subplots(figsize=(5.2, 4.4))

    kmf.fit(
        durations=high["surv_time"],
        event_observed=high["surv_event"],
        label="R-loop-program-high"
    )

    kmf.plot_survival_function(
        ax=ax,
        ci_show=False,
        color="#1f77b4",
        linewidth=1.8
    )

    kmf.fit(
        durations=low["surv_time"],
        event_observed=low["surv_event"],
        label="R-loop-program-low"
    )

    kmf.plot_survival_function(
        ax=ax,
        ci_show=False,
        color="#ff7f0e",
        linewidth=1.8
    )

    lr = logrank_test(
        high["surv_time"],
        low["surv_time"],
        event_observed_A=high["surv_event"],
        event_observed_B=low["surv_event"]
    )

    ax.text(
        0.05,
        0.08,
        f"Log-rank P = {lr.p_value:.3g}",
        transform=ax.transAxes,
        fontsize=11
    )

    ax.set_xlabel(
        "Time",
        fontsize=11
    )

    ax.set_ylabel(
        "Overall survival probability",
        fontsize=11
    )

    ax.set_title(
        "Survival of TCGA-LIHC patients stratified by RLP3/RLP4 score",
        fontsize=12,
        fontweight="bold"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        labelsize=10
    )

    ax.grid(False)

    plt.tight_layout()

    save_fig(
        fig,
        "Figure5C_TCGA_survival_RLP3_RLP4_high_low",
        5.2,
        4.4
    )

    plt.show()

    # ========================================================
    # Cox regression
    # ========================================================

    cox_df = surv[
        [
            "surv_time",
            "surv_event",
            "RLP3_RLP4_combined_score"
        ]
    ].copy()

    cph = CoxPHFitter()

    cph.fit(
        cox_df,
        duration_col="surv_time",
        event_col="surv_event"
    )

    cox_outfile = os.path.join(
        OUTDIR,
        "Figure5C_Cox_RLP3_RLP4_combined_score.csv"
    )

    cph.summary.to_csv(cox_outfile)

    print("Cox regression result saved to:")
    print(cox_outfile)

    display(cph.summary)

else:

    print("No suitable survival columns found. Please check clinical table.")

In [ ]:
# ============================================================
# 6. Immune / TME gene signatures
# ============================================================

immune_signatures = {
    "CD8 T cell": ["CD8A", "CD8B", "GZMB", "GZMH", "PRF1", "NKG7"],
    "Cytotoxicity": ["GZMB", "GZMH", "PRF1", "NKG7", "GNLY", "IFNG"],
    "Treg": ["FOXP3", "IL2RA", "CTLA4", "IKZF2", "TIGIT"],
    "Exhaustion": ["PDCD1", "CTLA4", "LAG3", "HAVCR2", "TIGIT", "TOX"],
    "TAM / Myeloid": ["C1QA", "C1QB", "C1QC", "LST1", "TYROBP", "FCER1G"],
    "M2 macrophage": ["CD163", "MRC1", "MSR1", "IL10", "TGFB1"],
    "CAF": ["COL1A1", "COL1A2", "DCN", "LUM", "ACTA2", "TAGLN"],
    "Antigen presentation": ["HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2"],
    "IFN response": ["ISG15", "IFIT1", "IFIT3", "MX1", "OAS1", "STAT1", "IRF7"]
}

immune_score = pd.DataFrame(index=expr_t.index)

for sig, genes in immune_signatures.items():
    matched = [g for g in genes if g in expr_t.columns]
    print(sig, len(matched), matched)

    if len(matched) >= 2:
        X = expr_t[matched]
        Xz = pd.DataFrame(
            StandardScaler().fit_transform(X),
            index=X.index,
            columns=X.columns
        )
        immune_score[sig] = Xz.mean(axis=1)

immune_score = immune_score.join(tcga_score[["RLP3_RLP4_combined_score", "Rloop_program_group"]])

immune_score.to_csv(
    os.path.join(FIG5_DIR, "Figure5D_TCGA_immune_signature_scores.csv")
)

In [ ]:
# ============================================================
# Figure5D
# Bubble plot: TCGA bulk immune-context validation
# RLP3/RLP4 score vs immune/TME signatures
# ============================================================

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ------------------------------------------------------------
# 1. Calculate correlations
# ------------------------------------------------------------

corr_rows = []

for sig in immune_signatures.keys():
    if sig not in immune_score.columns:
        continue

    tmp_sig = immune_score[[sig, "RLP3_RLP4_combined_score"]].dropna()

    if tmp_sig.shape[0] < 10:
        continue

    r, p = spearmanr(
        tmp_sig["RLP3_RLP4_combined_score"],
        tmp_sig[sig]
    )

    corr_rows.append({
        "Signature": sig,
        "Spearman_r": r,
        "P_value": p,
        "minus_log10_p": -np.log10(p + 1e-300)
    })

corr_df = pd.DataFrame(corr_rows)

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

corr_df["Significance"] = corr_df["P_value"].apply(p_to_star)

# 按生物学逻辑排序
sig_order = [
    "CD8 T cell",
    "Cytotoxicity",
    "Treg",
    "Exhaustion",
    "TAM / Myeloid",
    "M2 macrophage",
    "CAF",
    "Antigen presentation",
    "IFN response"
]

sig_order = [s for s in sig_order if s in corr_df["Signature"].values]

corr_df["Signature"] = pd.Categorical(
    corr_df["Signature"],
    categories=sig_order,
    ordered=True
)

corr_df = corr_df.sort_values("Signature").reset_index(drop=True)

corr_df.to_csv(
    os.path.join(FIG5_DIR, "Figure5D_RLP3_RLP4_immune_correlation_bubble.csv"),
    index=False
)

corr_df

In [ ]:
# ============================================================
# Figure5D
# Improved bubble plot
# Adobe Illustrator editable text
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Make text editable in Adobe Illustrator
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# Save function
# ============================================================

def save_fig(fig, name, width=6.2, height=4.8, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(
        os.path.join(outdir, f"{name}.pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.svg"),
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(outdir, f"{name}.png"),
        dpi=600,
        bbox_inches="tight"
    )

# ============================================================
# Prepare plotting dataframe
# ============================================================

plot_df = corr_df.copy()

plot_df = plot_df.sort_values(
    "Spearman_r",
    ascending=True
).reset_index(drop=True)

# ============================================================
# Bubble size
# ============================================================

size_min = 120
size_max = 650

p_min = plot_df["minus_log10_p"].min()
p_max = plot_df["minus_log10_p"].max()

if p_max == p_min:

    plot_df["bubble_size"] = (size_min + size_max) / 2

else:

    plot_df["bubble_size"] = (
        size_min
        + (plot_df["minus_log10_p"] - p_min)
        / (p_max - p_min)
        * (size_max - size_min)
    )

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 4.8))

sc = ax.scatter(
    x=plot_df["Spearman_r"],
    y=plot_df["Signature"],
    s=plot_df["bubble_size"],
    c=plot_df["Spearman_r"],
    cmap="RdBu_r",
    vmin=-0.25,
    vmax=0.25,
    edgecolors="black",
    linewidths=0.5,
    alpha=0.9
)

# ============================================================
# Zero reference line
# ============================================================

ax.axvline(
    x=0,
    color="grey",
    linestyle="--",
    linewidth=1
)

# ============================================================
# Add correlation labels
# ============================================================

for _, row in plot_df.iterrows():

    offset = 0.018 if row["Spearman_r"] >= 0 else -0.018
    ha = "left" if row["Spearman_r"] >= 0 else "right"

    ax.text(
        row["Spearman_r"] + offset,
        row["Signature"],
        f"{row['Spearman_r']:.2f} {row['Significance']}",
        ha=ha,
        va="center",
        fontsize=9
    )

# ============================================================
# Colorbar
# ============================================================

cbar = plt.colorbar(
    sc,
    ax=ax,
    fraction=0.045,
    pad=0.04
)

cbar.set_label(
    "Spearman r",
    fontsize=10
)

cbar.ax.tick_params(
    labelsize=9
)

# ============================================================
# Labels and title
# ============================================================

ax.set_xlim(-0.30, 0.30)

ax.set_xlabel(
    "Spearman correlation with RLP3/RLP4 combined score",
    fontsize=11
)

ax.set_ylabel("")

ax.set_title(
    "Bulk immune context associated with RLP3/RLP4 score",
    fontsize=13,
    fontweight="bold"
)

# ============================================================
# Formatting
# ============================================================

ax.tick_params(
    axis="x",
    labelsize=10
)

ax.tick_params(
    axis="y",
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(False)

plt.tight_layout()

# ============================================================
# Save figure
# ============================================================

save_fig(
    fig,
    "Figure5D_TCGA_RLP3_RLP4_immune_correlation_bubble_improved",
    6.2,
    4.8
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# Figure5E
# Immune checkpoint expression in R-loop-program-high vs low
# ============================================================

from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

checkpoint_genes = [
    "PDCD1", "CD274", "PDCD1LG2",
    "CTLA4", "LAG3", "HAVCR2", "TIGIT",
    "TOX", "ENTPD1", "IDO1"
]

checkpoint_genes = [g for g in checkpoint_genes if g in expr_t.columns]

ck_df = expr_t[checkpoint_genes].copy()
ck_df = ck_df.join(tcga_score[["Rloop_program_group"]])

ck_long = ck_df.reset_index().melt(
    id_vars=["index", "Rloop_program_group"],
    value_vars=checkpoint_genes,
    var_name="Gene",
    value_name="Expression"
)

ck_long = ck_long.rename(columns={"index": "Sample"})

ck_long["Rloop_program_group"] = pd.Categorical(
    ck_long["Rloop_program_group"],
    categories=["R-loop-program-low", "R-loop-program-high"],
    ordered=True
)

# ------------------------------------------------------------
# 1. Statistics: high vs low for each checkpoint gene
# ------------------------------------------------------------

stat_rows = []

for gene in checkpoint_genes:
    tmp_gene = ck_long[ck_long["Gene"] == gene].dropna()

    high = tmp_gene.loc[
        tmp_gene["Rloop_program_group"] == "R-loop-program-high",
        "Expression"
    ]

    low = tmp_gene.loc[
        tmp_gene["Rloop_program_group"] == "R-loop-program-low",
        "Expression"
    ]

    if len(high) > 0 and len(low) > 0:
        stat, pval = mannwhitneyu(high, low, alternative="two-sided")
    else:
        stat, pval = np.nan, np.nan

    stat_rows.append({
        "Gene": gene,
        "High_n": len(high),
        "Low_n": len(low),
        "High_median": high.median(),
        "Low_median": low.median(),
        "MannWhitney_U": stat,
        "P_value": pval
    })

stat_df = pd.DataFrame(stat_rows)

def p_to_star(p):
    if pd.isna(p):
        return "NA"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stat_df["Significance"] = stat_df["P_value"].apply(p_to_star)

stat_df.to_csv(
    os.path.join(FIG5_DIR, "Figure5E_checkpoint_high_vs_low_statistics.csv"),
    index=False
)

stat_df

In [ ]:
# ============================================================
# Figure5E
# Immune checkpoint expression: high vs low
# Adobe Illustrator editable text
# ============================================================

import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/05_Figure5_TCGA_LIHC"
os.makedirs(OUTDIR, exist_ok=True)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.unicode_minus"] = False

def save_fig(fig, name, width=11, height=5, outdir=OUTDIR):

    fig.set_size_inches(width, height)

    fig.savefig(os.path.join(outdir, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(outdir, f"{name}.svg"), bbox_inches="tight")
    fig.savefig(os.path.join(outdir, f"{name}.png"), dpi=600, bbox_inches="tight")

palette = {
    "R-loop-program-low": "#F28E2B",
    "R-loop-program-high": "#4E79A7"
}

fig, ax = plt.subplots(figsize=(11, 5))

sns.boxplot(
    data=ck_long,
    x="Gene",
    y="Expression",
    hue="Rloop_program_group",
    order=checkpoint_genes,
    hue_order=["R-loop-program-low", "R-loop-program-high"],
    palette=palette,
    width=0.42,
    dodge=0.82,
    fliersize=0,
    linewidth=1.15,
    ax=ax
)

sns.stripplot(
    data=ck_long,
    x="Gene",
    y="Expression",
    hue="Rloop_program_group",
    order=checkpoint_genes,
    hue_order=["R-loop-program-low", "R-loop-program-high"],
    dodge=0.82,
    palette=palette,
    size=1.8,
    alpha=0.22,
    jitter=0.17,
    linewidth=0,
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    title="",
    fontsize=11
)

ymax_by_gene = ck_long.groupby("Gene")["Expression"].max().to_dict()

global_ymin = ck_long["Expression"].min()
global_ymax = ck_long["Expression"].max()
yrange = global_ymax - global_ymin

for i, gene in enumerate(checkpoint_genes):

    sig = stat_df.loc[
        stat_df["Gene"] == gene,
        "Significance"
    ].values[0]

    y = ymax_by_gene[gene] + yrange * 0.07

    x1 = i - 0.24
    x2 = i + 0.24

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + yrange * 0.025, y + yrange * 0.025, y],
        color="black",
        linewidth=0.9
    )

    ax.text(
        i,
        y + yrange * 0.04,
        sig,
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_xlabel("")

ax.set_ylabel(
    "Expression, log2(CPM + 1)",
    fontsize=12
)

ax.set_title(
    "Immune checkpoint expression in TCGA-LIHC",
    fontsize=15,
    fontweight="bold"
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=11
)

ax.tick_params(
    axis="y",
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(False)

ax.set_ylim(
    global_ymin - yrange * 0.05,
    global_ymax + yrange * 0.24
)

plt.tight_layout()

save_fig(
    fig,
    "Figure5E_TCGA_checkpoint_high_vs_low_beautiful",
    11,
    5
)

plt.show()

print("Figure saved to:")
print(OUTDIR)

In [ ]:
# ============================================================
# 7. Functional pathway signatures in TCGA-LIHC
# ============================================================

pathway_signatures = {
    "Cell cycle": [
        "MKI67", "TOP2A", "UBE2C", "CDK1", "CCNB1", "CCNB2",
        "CDC20", "AURKA", "AURKB", "BIRC5", "PCNA"
    ],
    "DNA repair": [
        "BRCA1", "BRCA2", "RAD51", "FANCI", "FANCD2",
        "CHEK1", "CHEK2", "ATM", "ATR", "PARP1"
    ],
    "Hypoxia": [
        "HIF1A", "VEGFA", "CA9", "SLC2A1", "LDHA", "ENO1",
        "PGK1", "ANGPTL4", "BNIP3"
    ],
    "Glycolysis": [
        "HK2", "PFKP", "ALDOA", "GAPDH", "PGK1", "ENO1",
        "PKM", "LDHA", "SLC2A1"
    ],
    "EMT": [
        "VIM", "FN1", "SNAI1", "SNAI2", "ZEB1", "ZEB2",
        "TWIST1", "CDH2", "ITGA5"
    ],
    "IFN response": [
        "ISG15", "IFIT1", "IFIT3", "MX1", "OAS1",
        "STAT1", "IRF7", "CXCL10"
    ],
    "Antigen presentation": [
        "HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2",
        "NLRC5"
    ],
    "Myeloid inflammation": [
        "C1QA", "C1QB", "C1QC", "LST1", "TYROBP",
        "FCER1G", "AIF1", "CSF1R"
    ]
}

pathway_score = pd.DataFrame(index=expr_t.index)

for sig, genes in pathway_signatures.items():
    matched = [g for g in genes if g in expr_t.columns]
    print(sig, len(matched), matched)

    if len(matched) >= 2:
        X = expr_t[matched]
        Xz = pd.DataFrame(
            StandardScaler().fit_transform(X),
            index=X.index,
            columns=X.columns
        )
        pathway_score[sig] = Xz.mean(axis=1)

pathway_score = pathway_score.join(
    tcga_score[["RLP3_RLP4_combined_score", "Rloop_program_group"]]
)

pathway_score.to_csv(
    os.path.join(FIG5_DIR, "Figure5F_TCGA_pathway_signature_scores.csv")
)

In [ ]:
# ============================================================
# 8. Summary table for Figure5
# ============================================================

summary_files = {
    "RLP3_genes_matched": rlp3_matched,
    "RLP4_genes_matched": rlp4_matched,
    "checkpoint_genes_used": checkpoint_genes,
    "immune_signatures": list(immune_score.columns),
    "pathway_signatures": pathway_cols
}

with open(os.path.join(FIG5_DIR, "Figure5_summary.txt"), "w") as f:
    for k, v in summary_files.items():
        f.write(f"\n{k}\n")
        f.write("=" * 60 + "\n")
        if isinstance(v, list):
            f.write("\n".join(map(str, v)))
        else:
            f.write(str(v))
        f.write("\n")

print("Figure5 finished.")
print("Output directory:", FIG5_DIR)